In [ ]:
from google.colab import drive
drive.mount("/content/drive")

PROJECT_DIR = "/content/drive/MyDrive/NutriChat-RAG/NutriChat"
%cd "$PROJECT_DIR"



Mounted at /content/drive
/content/drive/MyDrive/NutriChat-RAG/NutriChat


In [ ]:
!pip install -r requirements.txt
!pip install -e .

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 86.2 MB/s eta 0:00:00
Obtaining file:///content/drive/MyDrive/NutriChat-RAG/NutriChat
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for nutrichat (pyproject.toml) ... done
  Created wheel for nutrichat: filename=nutrichat-0.1.0-0.editable-py3-none-any.whl size=2811 sha256=ee64a7b27e34589633c289f44cf3c9582946510f4f9fb0fca98072a11cca32a1
  Stored in directory: /tmp/pip-ephem-wheel-cache-qt409m7f/wheels/13/1f/4c/1f42e06f0c3ace164fb08724e739b140b59d7d29b5214e49d4
Successfully built nutrichat


In [ ]:
import torch
import pandas as pd
import numpy as np
from pathlib import Path

from sentence_transformers import util

from nutrichat.config import (
    EMBEDDING_MODEL,
    MIN_TOKEN_LENGTH,
    PDF_PAGE_OFFSET,
)

from nutrichat.data import open_and_read_pdf, load_eval_questions
from nutrichat.chunking import (
    add_sentences_to_pages,
    build_chunks,
    filter_chunks_by_min_tokens,
    add_chunk_ids,
)
from nutrichat.embeddings import (
    load_embedding_model,
    embed_query,
    embed_chunks,
)
from nutrichat.evaluation import page_hit_at_k, reciprocal_rank

In [ ]:
CHUNKING_STRATEGIES = [
    "sentence_15_no_overlap",
    "sentence_10_overlap_5",
    "sentence_8_overlap_4",
    "sentence_6_overlap_3",
    "word_180_overlap_40",
    "word_280_overlap_60",
]

In [ ]:
PDF_PATH = "data/nutrition_textbook.pdf"
EVAL_PATH = "data/eval_dataset_200_v2.json"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

pages = open_and_read_pdf(
    pdf_path=PDF_PATH,
    page_offset=PDF_PAGE_OFFSET,
)

pages = add_sentences_to_pages(pages)

eval_questions = load_eval_questions(EVAL_PATH)

# Keep retrieval evaluation focused on answerable questions.
answerable_questions = [
    q for q in eval_questions
    if q.get("answerable") is True and q.get("expected_behavior") == "answer"
]

print("Answerable questions:", len(answerable_questions))

Reading PDF:   0%|          | 0/894 [00:00<?, ?it/s]

Sentence splitting:   0%|          | 0/894 [00:00<?, ?it/s]

Answerable questions: 140


In [ ]:
embedding_model = load_embedding_model(
    model_name=EMBEDDING_MODEL,
    device=DEVICE,
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
def retrieve_top_k_for_strategy(
    query: str,
    chunks: list[dict],
    embeddings: torch.Tensor,
    embedding_model,
    k: int = 3,
):
    query_embedding = embed_query(
        query=query,
        model=embedding_model,
    ).to(embeddings.device)

    dot_scores = util.dot_score(query_embedding, embeddings)[0]

    k = min(k, len(chunks))

    scores, indices = torch.topk(dot_scores, k=k)

    context_items = []

    for score, idx in zip(scores, indices):
        item = chunks[int(idx)].copy()
        item["score"] = float(score.detach().cpu())
        context_items.append(item)

    return context_items

In [ ]:
rows = []

for strategy_name in CHUNKING_STRATEGIES:
    print("=" * 80)
    print("Testing strategy:", strategy_name)

    raw_chunks = build_chunks(
        pages=pages,
        strategy_name=strategy_name,
    )

    chunks = filter_chunks_by_min_tokens(
        chunks=raw_chunks,
        min_token_length=MIN_TOKEN_LENGTH,
    )

    chunks = add_chunk_ids(chunks)

    embeddings_np = embed_chunks(
    chunks=chunks,
    model=embedding_model,
    batch_size=64,
    show_progress_bar=True,
  )

    assert isinstance(
        embeddings_np,
        np.ndarray,
    ), f"Expected NumPy array, received {type(embeddings_np)}"

    assert embeddings_np.ndim == 2, (
        f"Expected a 2D embedding matrix, "
        f"received shape {embeddings_np.shape}"
    )

    assert embeddings_np.shape[0] == len(chunks), (
        f"Chunk and embedding counts do not match: "
        f"{len(chunks)} chunks versus "
        f"{embeddings_np.shape[0]} embeddings"
    )

    embeddings = torch.as_tensor(
        embeddings_np,
        dtype=torch.float32,
        device=DEVICE,
    )

    print("Number of chunks:", len(chunks))
    print("Embedding matrix:", embeddings.shape)

    avg_chunk_tokens = float(
        np.mean([chunk["chunk_token_count"] for chunk in chunks])
    )

    for question in answerable_questions:
        context_items = retrieve_top_k_for_strategy(
            query=question["question"],
            chunks=chunks,
            embeddings=embeddings,
            embedding_model=embedding_model,
            k=3,
        )

        retrieved_pages = [
            item.get("page_number", "Unknown")
            for item in context_items
        ]

        expected_pages = question.get("expected_pages", [])

        rows.append(
            {
                "strategy": strategy_name,
                "id": question["id"],
                "question": question["question"],
                "expected_pages": expected_pages,
                "retrieved_pages": retrieved_pages,
                "retrieved_scores": [
                    item["score"] for item in context_items
                ],
                "top_score": context_items[0]["score"] if context_items else 0.0,
                "page_hit_at_3": page_hit_at_k(
                    retrieved_pages=retrieved_pages,
                    expected_pages=expected_pages,
                ),
                "mrr": reciprocal_rank(
                    retrieved_pages=retrieved_pages,
                    expected_pages=expected_pages,
                ),
                "num_chunks": len(chunks),
                "avg_chunk_tokens": avg_chunk_tokens,
            }
        )

chunking_retrieval_df = pd.DataFrame(rows)
chunking_retrieval_df.head()

Testing strategy: sentence_15_no_overlap


sentence_15_no_overlap:   0%|          | 0/894 [00:00<?, ?it/s]

Batches:   0%|          | 0/19 [00:00<?, ?it/s]

Number of chunks: 1212
Embedding matrix: torch.Size([1212, 384])
Testing strategy: sentence_10_overlap_5


sentence_10_overlap_5:   0%|          | 0/894 [00:00<?, ?it/s]

...

Number of chunks: 1800
Embedding matrix: torch.Size([1800, 384])
Testing strategy: word_280_overlap_60


word_280_overlap_60:   0%|          | 0/894 [00:00<?, ?it/s]

Batches:   0%|          | 0/21 [00:00<?, ?it/s]

Number of chunks: 1309
Embedding matrix: torch.Size([1309, 384])


,strategy,id,question,expected_pages,retrieved_pages,retrieved_scores,top_score,page_hit_at_3,mrr,num_chunks,avg_chunk_tokens
0,sentence_15_no_overlap,A001,Which six classes of nutrients are required fo...,[4],"[4, 10, 9]","[0.8033629655838013, 0.7745763659477234, 0.754...",0.803363,True,1.0,1212,276.644596
1,sentence_15_no_overlap,A002,What is a macronutrient?,[5],"[5, 7, 477]","[0.7920447587966919, 0.7379722595214844, 0.707...",0.792045,True,1.0,1212,276.644596
2,sentence_15_no_overlap,A003,What does the term Calorie on a Nutrition Fact...,"[264, 735, 726, 5]","[264, 726, 735]","[0.7435973882675171, 0.7383255958557129, 0.733...",0.743597,True,1.0,1212,276.644596
3,sentence_15_no_overlap,A004,Why is water considered a required macronutrie...,"[5, 7]","[7, 5, 265]","[0.8010458946228027, 0.7867315411567688, 0.738...",0.801046,True,1.0,1212,276.644596
4,sentence_15_no_overlap,A005,Why are nutrient-dense foods considered higher...,[13],"[554, 36, 266]","[0.7913213968276978, 0.7610929012298584, 0.751...",0.791321,False,0.0,1212,276.644596


In [ ]:
chunking_summary = (
    chunking_retrieval_df
    .groupby("strategy")
    .agg(
        num_chunks=("num_chunks", "first"),
        avg_chunk_tokens=("avg_chunk_tokens", "first"),
        page_hit_at_3=("page_hit_at_3", "mean"),
        mrr=("mrr", "mean"),
        avg_top_score=("top_score", "mean"),
    )
    .reset_index()
    .sort_values(
        by=["page_hit_at_3", "mrr", "avg_top_score"],
        ascending=False,
    )
)

chunking_summary

,strategy,num_chunks,avg_chunk_tokens,page_hit_at_3,mrr,avg_top_score
2,sentence_6_overlap_3,4192,137.979902,0.950000,0.902381,0.814236
4,word_180_overlap_40,1800,221.428889,0.950000,0.861905,0.796240
3,sentence_8_overlap_4,3258,171.105893,0.942857,0.885714,0.808623
0,sentence_10_overlap_5,2684,200.045082,0.928571,0.879762,0.803481
1,sentence_15_no_overlap,1212,276.644596,0.921429,0.869048,0.790519
5,word_280_overlap_60,1309,289.385791,0.914286,0.839286,0.791977


In [ ]:
Path("results/chunking_ablation").mkdir(parents=True, exist_ok=True)

chunking_retrieval_df.to_csv(
    "results/chunking_ablation/chunking_retrieval_rows.csv",
    index=False,
)

chunking_summary.to_csv(
    "results/chunking_ablation/chunking_retrieval_summary.csv",
    index=False,
)